In [3]:
# Nuestro proyecto trabaja en 3 niveles
# y cada uno necesita reglas distintas:

# 1. Episodio (fila) → una internación en un hospital
# 2. Transición (arista) → movimiento entre hospitales
# 3. Paciente (trayectoria completa) → historia total


# en este archivo:
# BASE 1 — df_base_limpia (episodios)
# - Objetivo: Tener todas las internaciones válidas, sin romper historias

# - ERROR en tu enfoque actual: Estás filtrando pacientes enteros (pacientes_final_ok), cuando acá deberías: NO eliminar pacientes todavía

# - Qué limpiar acá (solo cosas “objetivamente malas”)

#             - filas con:

#             fechas inválidas (fecha_ingreso > fecha_egreso)
#             hospitales missing
#             edad absurda (ej: <0 o >120)
#             motivos claramente basura (error, quizás nan dependiendo contexto)

#             - estandarización:

#             nombres hospitales
#             fechas
#             motivos (lowercase, strip)

#                     PERO:
#                     ❌ NO usar desenlace todavía
#                     ❌ NO eliminar pacientes completos
#                     ❌ NO usar prioridad clínica todavía

In [4]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import ast
import pandas as pd
import numpy as np
from src.config import *

In [5]:
# 1. CARGA Y RENOMBRE COMPLETO (Recuperado de tu código original)
# ==============================================================================
df_raw = pd.read_excel("../data/pacientes.xlsx")
hospitales = pd.read_csv("../data/hospitales_coordenadas.csv")

dict_comp = dict(zip(hospitales['Nombre Hospital'], hospitales['complejidad']))
hospitales['color_rgb'] = hospitales['color'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

df_base = df_raw.rename(columns={
    'Id Hospital': 'hospital_id', 'Nombre Hospital': 'hospital_origen',
    'Id': 'paciente_id', 'Fecha inicio': 'fecha_ingreso', 'Fecha egreso': 'fecha_egreso',
    'Estado al ingreso': 'estado_ingreso', 'Tipo al ingreso': 'tipo_ingreso',
    'Último estado': 'estado_ultimo', 'Último tipo': 'tipo_ultimo',
    'Sexo': 'sexo', 'Edad': 'edad', 'Nivel riesgo clínico': 'riesgo_clinico',
    'Nivel riesgo social': 'riesgo_social', 'Enfermedades preexistentes Covid-19': 'comorbilidades_covid',
    'Enfermedades preexistentes pediatría': 'comorbilidades_pediatria', 'Vacuna': 'vacuna',
    'Cant. dosis': 'cantidad_dosis', '1º dosis': 'fecha_dosis_1', '2º dosis': 'fecha_dosis_2',
    'Buscado en el ministerio': 'validado_ministerio', 'Obra social': 'obra_social',
    'Asistencia Respiratoria Mecánica': 'requiere_arm', 'Motivo': 'motivo_egreso',
    'Operación': 'operacion', 'Última actualización': 'fecha_ultima_actualizacion',
    'Pasó por Críticas': 'paso_criticas', 'Pasó por Intermedias': 'paso_intermedias',
    'Pasó por Generales': 'paso_generales'
}).copy()

df_base['hospital_origen'] = df_base['hospital_origen'].replace({
    'Módulo Hospitalario 11- FV': 'Módulo Hospitalario 11 - FV',
    'Módulo Hospitalario  9 - AB': 'Módulo Hospitalario 9 - AB'
}).str.strip()


df_base['fecha_ingreso'] = pd.to_datetime(df_base['fecha_ingreso'], errors='coerce')
df_base['fecha_egreso'] = pd.to_datetime(df_base['fecha_egreso'], errors='coerce')
df_base['edad'] = pd.to_numeric(df_base['edad'], errors='coerce')
df_base['dias_en_nodo'] = (
    (df_base['fecha_egreso'] - df_base['fecha_ingreso'])
    .dt.days
)

df_base = df_base.sort_values(['paciente_id', 'fecha_ingreso'])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import pandas as pd
import numpy as np

# CONFIGURACIÓN GLOBAL "MODO NOCHE"
plt.style.use('dark_background')
color_fondo = '#0a0a0c'
color_texto = '#A0A0B0'
color_cyan = '#00FFFF'
color_magenta = '#FF007F'

# Asegurarse de tener el df de traslados armado (como en el código anterior)
df_graf = df_base_limpia.sort_values(['paciente_id', 'fecha_ingreso']).copy()
df_graf['hospital_destino'] = df_graf.groupby('paciente_id')['hospital_origen'].shift(-1)
df_traslados = df_graf.dropna(subset=['hospital_destino'])
df_traslados = df_traslados[df_traslados['hospital_origen'] != df_traslados['hospital_destino']]


# ==============================================================================
# GRÁFICO 1: LA COMPLEJIDAD DEL SISTEMA (Grafo de Red Sin Etiquetas)
# Muestra: "Análisis de Redes Complexas"
# ==============================================================================
df_rutas = df_traslados.groupby(['hospital_origen', 'hospital_destino']).size().reset_index(name='peso')
G = nx.DiGraph()
for _, row in df_rutas.iterrows():
    G.add_edge(row['hospital_origen'], row['hospital_destino'], weight=row['peso'])

fig1, ax1 = plt.subplots(figsize=(12, 12), dpi=150)
fig1.patch.set_facecolor(color_fondo)
ax1.set_facecolor(color_fondo)

pos = nx.spring_layout(G, k=0.9, iterations=100, seed=42)
pesos = [G[u][v]['weight'] for u, v in G.edges()]
max_peso = max(pesos) if pesos else 1
grosores_glow = [(w / max_peso) * 6 + 0.5 for w in pesos]

# Nodos genéricos brillantes (sin asociar a hospitales específicos)
grados = [G.degree(nodo) for nodo in G.nodes()]
node_sizes = [g * 30 + 50 for g in grados]

nx.draw_networkx_edges(G, pos, edge_color=color_cyan, width=grosores_glow, alpha=0.3, ax=ax1, arrows=False)
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=color_magenta, alpha=0.8, ax=ax1)

ax1.axis('off')
ax1.set_title("Topología de Derivaciones Hospitalarias", color=color_texto, fontsize=16, pad=20)
fig1.tight_layout()
fig1.savefig("../data/final_data/linkedin_01_red.png", facecolor=color_fondo, dpi=300)
plt.close(fig1)


# ==============================================================================
# GRÁFICO 2: IDENTIFICANDO PATRONES (Heatmap de Traslados por Día)
# Muestra: "Dinámicas y Cuellos de Botella" (Estacionalidad abstracta)
# ==============================================================================
df_traslados['dia_semana'] = df_traslados['fecha_ingreso'].dt.day_name()
df_traslados['mes'] = df_traslados['fecha_ingreso'].dt.month

# Ordenar días
dias_orden = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heatmap_data = df_traslados.groupby(['dia_semana', 'mes']).size().unstack(fill_value=0)
heatmap_data = heatmap_data.reindex(dias_orden)

fig2, ax2 = plt.subplots(figsize=(10, 6), dpi=150)
fig2.patch.set_facecolor(color_fondo)
ax2.set_facecolor(color_fondo)

# Mapa de calor usando una paleta que va de negro a cyan neón
sns.heatmap(heatmap_data, cmap="crest_r", linewidths=0.5, linecolor=color_fondo, 
            cbar=False, ax=ax2)

ax2.set_title("Intensidad de Derivaciones (Patrón Temporal)", color=color_texto, fontsize=14, pad=15)
ax2.set_xlabel("Evolución Temporal ->", color=color_texto)
ax2.set_ylabel("")
ax2.tick_params(colors=color_texto, labelsize=10)
# Ocultar etiquetas exactas de los ejes para no spoilear el periodo exacto
ax2.set_xticks([]) 
ax2.set_yticks([]) 

fig2.tight_layout()
fig2.savefig("../data/final_data/linkedin_02_patrones.png", facecolor=color_fondo, dpi=300)
plt.close(fig2)


# ==============================================================================
# GRÁFICO 3: DISTRIBUCIÓN DE DÍAS (KDE de Estancias)
# Muestra: "Proponer métricas y exploración de datos"
# ==============================================================================
fig3, ax3 = plt.subplots(figsize=(10, 5), dpi=150)
fig3.patch.set_facecolor(color_fondo)
ax3.set_facecolor(color_fondo)

# Filtramos outliers extremos solo para que el gráfico quede estético
datos_estancia = df_base_limpia[df_base_limpia['dias_en_nodo'] < 30]['dias_en_nodo']

# Curva de densidad suave
sns.kdeplot(datos_estancia, color=color_magenta, fill=True, alpha=0.4, linewidth=2, ax=ax3)

ax3.set_title("Distribución de Tiempos de Internación por Episodio", color=color_texto, fontsize=14, pad=15)
ax3.set_xlabel("Días de Estancia (Escala Relativa)", color=color_texto)
ax3.set_ylabel("Densidad de Pacientes", color=color_texto)

# Limpiamos los ejes para mantener la confidencialidad
ax3.tick_params(colors=color_texto)
ax3.set_yticklabels([])
ax3.set_xticklabels([])
for spine in ax3.spines.values():
    spine.set_color('#333344')

fig3.tight_layout()
fig3.savefig("../data/final_data/linkedin_03_estancia.png", facecolor=color_fondo, dpi=300)
plt.close(fig3)

print("✅ ¡Los 3 gráficos se generaron en ../data/final_data/!")

✅ ¡Los 3 gráficos se generaron en ../data/final_data/!


### Extra: busqueda de errores o casos raros

In [ ]:
# pacientes que estuvieron la mayor cantidad de tiempo internados
# me llamo la atencion 500 dias asi que reviso esto..
cols = [
    'paciente_id',
    'hospital_origen',
    'fecha_ingreso',
    'fecha_egreso',
    'dias_en_nodo',
    'motivo_egreso',
    'tipo_egreso'
]

df_base_limpia.loc[
    df_base_limpia['dias_en_nodo'] == df_base_limpia['dias_en_nodo'].max(),
    cols
]

df_base_limpia.nlargest(10, 'dias_en_nodo')[cols]

(df_base_limpia['dias_en_nodo'] > 100).mean()

In [ ]:
# pacientes que estuvieron la mayor cantidad de tiempo internados
df_base_limpia.nsmallest(10, 'dias_en_nodo')[cols]
(df_base_limpia['dias_en_nodo'] < 1).mean()

In [ ]:
# distribución de duración
import matplotlib.pyplot as plt

# 1. Calculamos el límite del percentil (ej: 95%)
# Esto ignora el 5% de los casos con estancias extremadamente largas
limite_outliers = df_base_limpia['dias_en_nodo'].quantile(0.95)

# 2. Filtramos los datos
df_filtrado = df_base_limpia[df_base_limpia['dias_en_nodo'] <= limite_outliers]

# 3. Configuramos los bins para que cada uno sea un entero
valor_max = int(df_filtrado['dias_en_nodo'].max())
# Usamos np.arange para crear cortes cada 0.5 para que el entero quede en el centro
bins = np.arange(0, valor_max + 2) - 0.5

# 4. Graficamos
plt.figure(figsize=(12, 6))
plt.hist(df_filtrado['dias_en_nodo'], bins=bins, edgecolor='white', color='#3498db')

# Forzamos a que el eje X muestre los números de 1 en 1
plt.xticks(range(0, valor_max + 1))

plt.title(f'Distribución de Estancia (Corte en {limite_outliers:.1f} días)')
plt.xlabel('Días (cada barra es 1 día entero)')
plt.ylabel('Frecuencia (Nº de casos)')
plt.grid(axis='y', alpha=0.3)
plt.show()

print(f"El gráfico muestra el 95% de los datos. El valor máximo mostrado es {valor_max} días.")

In [ ]:
# 2 mins de diferencia egreso - ingreso

# Calcular la diferencia exacta en minutos
# .total_seconds() / 60 nos da la precisión que buscas
duracion_minutos = (df_base_limpia['fecha_egreso'] - df_base_limpia['fecha_ingreso']).dt.total_seconds() / 60

# Filtrar los que duran 2 minutos o menos (incluye negativos si los hay)
errores_tiempo = df_base_limpia[duracion_minutos <= 2]

# Resultados
total_errores = len(errores_tiempo)
porcentaje_errores = (total_errores / len(df_base_limpia)) * 100

print(f"Detectados {total_errores} registros con duración <= 2 minutos.")
print(f"Representan el {porcentaje_errores:.2f}% del total de la base.")

# Ver una muestra de esos errores para confirmar
print("\nMuestra de posibles errores:")
print(errores_tiempo[['fecha_ingreso', 'fecha_egreso', 'dias_en_nodo']].head())

In [ ]:
# pacientes con muchos episodios
df_base_limpia['paciente_id'].value_counts().head(10)

In [ ]:
# episodios que potencialmente son duplicados
df_base_limpia[
    df_base_limpia.duplicated(
        subset=['paciente_id', 'hospital_origen', 'fecha_ingreso'], ## obs, hay un problema grande con las fechas de ingreso.
        # Para mi se hacian en un momento dado del dia se actualizaban todas
        keep=False
    )
].sort_values(['paciente_id', 'fecha_ingreso']).head(10)

In [ ]:
# hospitales con duracion de estadias mas altos
df_base_limpia.groupby('hospital_origen')['dias_en_nodo'].mean().sort_values(ascending=False).head(10)

In [ ]:
# cuantos hay de cada tipo, para ver cuantos son administrativos
df_base_limpia['tipo_egreso'].value_counts(normalize=True)

In [ ]:
# columnas que NO podemos usar:
print(df_base_limpia.isna().mean().sort_values(ascending=False).head(10))

In [ ]:
# CATEGORÍA 1 y 2: FILTROS DUROS (Integridad Estructural y Lógica)
# ==============================================================================

# --- MASK 1: Integridad Estructural
mask_integridad_core = (
    df_base['paciente_id'].notna() &
    df_base['hospital_origen'].notna() &
    (df_base['hospital_origen'] != '') &
    df_base['fecha_egreso'].notna() &
    df_base['fecha_ingreso'].notna()
)

# --- MASK 2: Coherencia Lógica de Fechas (<= porque la precisión es por días)
mask_fechas_logicas = (
    (df_base['fecha_ingreso'] <= df_base['fecha_egreso'])
)

# --- FILTRO: Crear base limpia definitiva
df_base_limpia = df_base[
    mask_integridad_core &
    mask_fechas_logicas
].copy()


In [7]:
# CATEGORÍA 3: FLAGS SUAVES (Calidad de Datos Clínicos)
# ==============================================================================
# Se aplican sobre df_base_limpia porque ya pasaron el filtro duro

# edad fuera de rango razonable
df_base_limpia['flag_edad_rara'] = (
    (df_base_limpia['edad'] < 0) | (df_base_limpia['edad'] > 110)
)

# duración negativa o muy larga
df_base_limpia['flag_duracion_negativa'] = df_base_limpia['dias_en_nodo'] < 0
df_base_limpia['flag_duracion_larga'] = df_base_limpia['dias_en_nodo'] > 60  # ajustable


In [8]:
# CATEGORÍA 4: FLAGS DE SISTEMA (Validez Administrativa)
# ==============================================================================

# normalizar texto
df_base_limpia['motivo_egreso_clean'] = (
    df_base_limpia['motivo_egreso']
    .astype(str)
    .str.lower()
    .str.strip()
)

# flag administrativo inválido
df_base_limpia['flag_egreso_admin_invalido'] = df_base_limpia['motivo_egreso_clean'].str.contains(
    'anulado|error|duplicado', case=False, na=False
)

# clasificación simple (puede crecer después)
def clasificar_egreso(x):
    if pd.isna(x):
        return 'desconocido'
    x = str(x).lower()
    if 'muerte' in x:
        return 'muerte'
    elif 'alta-domiciliaria' in x:
        return 'alta'
    elif 'traslado-extra-sanitario' in x:
        return 'hotel'
    elif 'traslado-otro' in x:
        return 'hospital-externo'
    elif 'traslado-hospital-de-la-red' in x:
        return 'traslado'
    elif 'anulado' in x or 'otro' in x:
        return 'administrativo'
    elif 'traslado-extra-sanitario' in x:
        return 'hotel'
    else:
        return 'administrativo'

df_base_limpia['tipo_egreso'] = df_base_limpia['motivo_egreso'].apply(clasificar_egreso)


In [9]:
# 5. CONSTRUCCIÓN DE BASE LIMPIA ya realizada en el bloque de filtros duros.


In [10]:
# 6. CHECKS BÁSICOS (para que mires calidad sin romper nada)
# ==============================================================================

print("Total episodios crudos:", len(df_base))
print("Episodios válidos (tras Filtros Duros):", len(df_base_limpia))

print("\n% flag_edad_rara:", df_base_limpia['flag_edad_rara'].mean())
print("% sin fecha egreso (debe ser 0.0):", df_base_limpia['fecha_egreso'].isna().mean())
print("% flag_egreso_admin_invalido:", df_base_limpia['flag_egreso_admin_invalido'].mean())

print("\nDuración (días) resumen:")
print(df_base_limpia['dias_en_nodo'].describe())

print("\nTop hospitales:")
print(df_base_limpia['hospital_origen'].value_counts().head(10))


Total episodios: 29697
Episodios válidos: 24953

% edad rara: 0.004368212239009337
% sin fecha egreso: 0.0
% egreso administrativo: 0.022842944736103876

Duración (días) resumen:
count    24953.000000
mean         9.779265
std         18.124005
min          0.000000
25%          2.000000
50%          5.000000
75%         11.000000
max        553.000000
Name: dias_en_nodo, dtype: float64

Top hospitales:
hospital_origen
Mi Pueblo                      4267
Oñativia                       3555
UPA 5 - AB                     3019
Lucio Meléndez                 2788
Oller                          1658
Módulo Hospitalario 11 - FV    1604
UPA 11 - FV                    1541
Evita Pueblo                   1461
El Cruce                       1321
Módulo Hospitalario 10 - QU    1178
Name: count, dtype: int64


In [11]:
# esta base responde
# “qué internaciones existieron realmente”
# (o deberian. Para mi hay repetidos que no se como sacar)
    # # faltaria:
    # - resolver duplicados
    # - resolver superposiciones
    # - consistencia longitudinal por paciente

In [12]:
df_base_limpia.to_excel("../data/final_data/df_base_limpia.xlsx", index=False)
# df_base_limpia.to_parquet("../data/final_data/df_base_limpia.parquet", index=False)